# Lab 07 - Logistic Regression: Hazardous Event Classification

This notebook applies **Lab 7 - Logistic Regression** to the assignment's binary hazardous-event target.


## Logistic regression concepts used

- Train a binary logistic regression classifier.
- Use class weighting for an imbalanced target.
- Evaluate probability-based predictions with threshold tuning.
- Inspect coefficients for model interpretation.

The primary model excludes `European_AQI` to reduce leakage risk.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'PM2_5_ug_m3',
    'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score
)


In [ ]:
from sklearn.linear_model import LogisticRegression

model_df = latest_rows(add_time_features(data), 100000)
train_df, test_df = chronological_split(model_df, train_size=0.8)
X_train = train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['Hazardous_Event']
X_test = test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['Hazardous_Event']

logistic_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs'))
])
logistic_model.fit(X_train, y_train)
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = int(np.nanargmax(f1_scores))
best_threshold = thresholds[best_idx]
threshold_pred = (y_prob >= best_threshold).astype(int)
print('Default threshold F1:', f1_score(y_test, y_pred))
print('Best threshold:', round(best_threshold, 3))
print('Tuned threshold F1:', f1_score(y_test, threshold_pred))
print(classification_report(y_test, threshold_pred, digits=3))


In [ ]:
cm = confusion_matrix(y_test, threshold_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Not hazardous', 'Hazardous'],
            yticklabels=['Not hazardous', 'Hazardous'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic regression confusion matrix')
plt.show()


In [ ]:
feature_names = logistic_model.named_steps['preprocess'].get_feature_names_out()
coefs = logistic_model.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
coef_df.sort_values('abs_coefficient', ascending=False).head(15)


In [ ]:
plt.figure(figsize=(8, 5))
plot_df = coef_df.sort_values('abs_coefficient', ascending=False).head(12).sort_values('coefficient')
sns.barplot(data=plot_df, x='coefficient', y='feature')
plt.axvline(0, color='black', linewidth=1)
plt.title('Largest logistic regression coefficients')
plt.show()


## What was learned from logistic regression

Logistic regression is a strong interpretable baseline for hazardous-event risk. Coefficients help explain which scaled features increase or reduce predicted risk, while threshold tuning addresses the imbalance between hazardous and non-hazardous hours.
